# Notebook 7 (Fase 3) — Análisis de Sentimiento NLP (VADER) → MongoDB
### Ernesto Investing AI · iDeSo · UNMSM — Semana 13

**Objetivo:**

Obtener noticias financieras de los 5 tickers mineros del proyecto, clasificar su sentimiento con `NLTK VADER` (Bullish / Bearish / Neutral), y almacenar los resultados en MongoDB Atlas.

**Prerequisito:** Haber ejecutado el Notebook 1 (colección `precios_ohlcv` poblada). Este notebook no depende de esos datos para funcionar, pero comparte la misma base de datos del proyecto.

**Salida:** Colección `sentimiento_nlp` poblada para los 5 tickers y archivo `datos_nlp.json` listo para el frontend.

**Alcance:** Este notebook cubre únicamente el módulo de **Análisis de Sentimiento NLP (VADER)** de la Fase 3. El módulo de **LSTM Regressor** (pronóstico de precios) se documentará en un notebook independiente.

**Pipeline:** yfinance (noticias) → limpieza y tokenización → VADER → MongoDB (`sentimiento_nlp`) → JSON (`datos_nlp.json`)

In [1]:
# Paso 1 — Instalar librerias necesarias
!pip install "pymongo[srv]" yfinance nltk plotly --quiet

print("Librerias instaladas correctamente")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.5 MB/s eta 0:00:00
Librerias instaladas correctamente


In [2]:
# Paso 2 — Conectar a MongoDB Atlas
# La contrasena NUNCA se escribe en texto plano en el notebook: se pide de forma
# oculta con getpass y se inserta en la URI de conexion en tiempo de ejecucion.
# Alternativa con Secrets de Colab:
# from google.colab import userdata; MONGO_URI = userdata.get('MONGO_URI')
from getpass import getpass
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import pandas as pd
import numpy as np

DB_USER = "gdev03_db_user"
DB_PASSWORD = getpass("Password de MongoDB Atlas: ")

MONGO_URI = (
    f"mongodb+srv://{DB_USER}:{DB_PASSWORD}"
    "@cluster0.sxjck8h.mongodb.net/?appName=Cluster0"
)

client = MongoClient(MONGO_URI, server_api=ServerApi('1'))

# Verificar la conexion con un ping, manejando errores de forma explicita
try:
    client.admin.command("ping")
    print("Conectado a MongoDB Atlas correctamente")
except Exception as e:
    raise Exception("Ocurrio el siguiente error al conectar: ", e)

db = client["ernesto_investing_ai"]


Password de MongoDB Atlas: ··········
Conectado a MongoDB Atlas correctamente


In [3]:
# Paso 3 — Definir los 5 tickers mineros del proyecto
TICKERS = {
    "FSM": "Fortuna Silver Mines Inc.",
    "VOLCABC1.LM": "Volcan Compania Minera S.A.A.",
    "ABX.TO": "Barrick Gold Corporation",
    "BVN": "Compania de Minas Buenaventura",
    "BHP": "BHP Billiton Limited"
}

print("Tickers del proyecto:")
for t, nombre in TICKERS.items():
    print(f"  {t}: {nombre}")


Tickers del proyecto:
  FSM: Fortuna Silver Mines Inc.
  VOLCABC1.LM: Volcan Compania Minera S.A.A.
  ABX.TO: Barrick Gold Corporation
  BVN: Compania de Minas Buenaventura
  BHP: BHP Billiton Limited


In [4]:
# Paso 4 — Descargar recursos de NLTK y preparar el analizador VADER
import nltk

# vader_lexicon: diccionario de polaridad utilizado por VADER
# punkt / punkt_tab: modelos de tokenizacion de oraciones y palabras
nltk.download("vader_lexicon", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize

analizador = SentimentIntensityAnalyzer()
print("Lexicon VADER descargado y analizador listo")


Lexicon VADER descargado y analizador listo


In [5]:
# Paso 5 — Dataset de respaldo simulado (garantiza reproducibilidad)
# Estos titulares son generados por el propio notebook, no se extraen de ninguna
# fuente externa. Solo se usan cuando `.news` de yfinance no devuelve suficientes
# resultados (algo frecuente para tickers menos cubiertos por la prensa financiera).
import random
from datetime import datetime, timedelta

random.seed(42)

PLANTILLAS_POSITIVAS = [
    "{empresa} reports stronger than expected quarterly output",
    "Analysts upgrade {empresa} on rising commodity prices",
    "{empresa} announces new exploration project with promising results",
    "{empresa} shares rally after positive production guidance",
    "Investors optimistic about {empresa} following cost reduction plan",
]

PLANTILLAS_NEGATIVAS = [
    "{empresa} faces production delays amid regulatory concerns",
    "{empresa} shares fall after disappointing earnings report",
    "Analysts downgrade {empresa} citing rising operational costs",
    "{empresa} halts operations at key site due to safety review",
    "Market concerns grow over {empresa} debt levels",
]

PLANTILLAS_NEUTRAS = [
    "{empresa} to present at upcoming mining industry conference",
    "{empresa} releases scheduled quarterly production report",
    "{empresa} board announces routine management changes",
    "Market watchers await {empresa} earnings call next week",
    "{empresa} maintains previous production guidance for the year",
]

def generar_noticias_simuladas(ticker, nombre_empresa, n=6):
    """Genera un set reproducible de titulares simulados (positivos, negativos y neutros)."""
    plantillas = PLANTILLAS_POSITIVAS + PLANTILLAS_NEGATIVAS + PLANTILLAS_NEUTRAS
    seleccion = random.sample(plantillas, k=min(n, len(plantillas)))

    ahora = datetime.now()
    noticias = []
    for i, plantilla in enumerate(seleccion):
        noticias.append({
            "titulo": plantilla.format(empresa=nombre_empresa),
            "publicador": "Fuente simulada (respaldo)",
            "fecha_publicacion": (ahora - timedelta(days=i)).strftime("%Y-%m-%d"),
            "fuente": "simulado"
        })
    return noticias

print("Funcion generar_noticias_simuladas definida")


Funcion generar_noticias_simuladas definida


In [6]:
# Paso 6 — Obtener noticias reales de yfinance (con respaldo simulado)
import yfinance as yf

MIN_NOTICIAS = 5

def _extraer_campos_noticia(item):
    """Normaliza el formato de una noticia de yfinance.

    yfinance ha cambiado el formato de `.news` entre versiones: en versiones
    recientes cada elemento trae los datos anidados dentro de la clave
    'content'. Esta funcion soporta ambos formatos para mayor robustez.
    """
    contenido = item.get("content", item)

    titulo = contenido.get("title") or item.get("title")

    proveedor = contenido.get("provider")
    if isinstance(proveedor, dict):
        publicador = proveedor.get("displayName")
    else:
        publicador = item.get("publisher")

    fecha = contenido.get("pubDate") or item.get("providerPublishTime")

    if isinstance(fecha, (int, float)):
        fecha = datetime.fromtimestamp(fecha).strftime("%Y-%m-%d")
    elif isinstance(fecha, str) and fecha:
        fecha = fecha[:10]
    else:
        fecha = datetime.now().strftime("%Y-%m-%d")

    return titulo, publicador or "Desconocido", fecha


def obtener_noticias(ticker, nombre_empresa):
    """Intenta obtener noticias reales via yfinance; si son insuficientes, usa respaldo simulado."""
    noticias = []
    try:
        raw = yf.Ticker(ticker).news or []
        for item in raw:
            titulo, publicador, fecha = _extraer_campos_noticia(item)
            if titulo:
                noticias.append({
                    "titulo": titulo,
                    "publicador": publicador,
                    "fecha_publicacion": fecha,
                    "fuente": "yfinance"
                })
    except Exception as e:
        print(f"  [{ticker}] Aviso: fallo al obtener noticias de yfinance ({e})")

    if len(noticias) < MIN_NOTICIAS:
        faltantes = MIN_NOTICIAS - len(noticias)
        noticias += generar_noticias_simuladas(ticker, nombre_empresa, n=max(faltantes, 6))
        print(f"  [{ticker}] Noticias reales insuficientes, se completo con respaldo simulado")

    return noticias

print("Funcion obtener_noticias definida")


Funcion obtener_noticias definida


In [7]:
# Paso 7 — Preprocesamiento de texto: limpieza y tokenizacion
import re

def limpiar_texto(texto):
    """Limpieza basica: minusculas y eliminacion de caracteres no alfanumericos."""
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def tokenizar(texto_limpio):
    """Tokeniza el texto ya limpiado en palabras individuales."""
    return word_tokenize(texto_limpio)

# Prueba rapida de las funciones de preprocesamiento
ejemplo = "Barrick Gold shares RALLY after strong earnings!!"
ejemplo_limpio = limpiar_texto(ejemplo)
print(f"Texto original : {ejemplo}")
print(f"Texto limpio   : {ejemplo_limpio}")
print(f"Tokens         : {tokenizar(ejemplo_limpio)}")


Texto original : Barrick Gold shares RALLY after strong earnings!!
Texto limpio   : barrick gold shares rally after strong earnings
Tokens         : ['barrick', 'gold', 'shares', 'rally', 'after', 'strong', 'earnings']


In [8]:
# Paso 8 — Analisis de sentimiento con VADER y clasificacion Bullish/Bearish/Neutral
def analizar_sentimiento(texto):
    """Aplica VADER sobre el texto limpio y retorna los scores compound, pos, neg, neu."""
    texto_limpio = limpiar_texto(texto)
    scores = analizador.polarity_scores(texto_limpio)
    return {
        "compound": round(scores["compound"], 4),
        "pos": round(scores["pos"], 4),
        "neg": round(scores["neg"], 4),
        "neu": round(scores["neu"], 4),
    }

def clasificar_sentimiento(compound):
    """Clasifica el sentimiento segun el umbral estandar de VADER."""
    if compound > 0.05:
        return "Bullish"
    elif compound < -0.05:
        return "Bearish"
    else:
        return "Neutral"

print("Funciones analizar_sentimiento y clasificar_sentimiento definidas")


Funciones analizar_sentimiento y clasificar_sentimiento definidas


In [9]:
# Paso 9 — Procesar un ticker completo: noticias -> VADER -> MongoDB
def limpiar_valor(v):
    """Reemplaza valores NaN/invalidos por None para asegurar documentos JSON validos."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except TypeError:
        pass
    return v


def procesar_ticker_nlp(ticker, nombre_empresa):
    """Obtiene noticias, aplica VADER a cada titular, calcula el consolidado y guarda en MongoDB."""
    noticias_raw = obtener_noticias(ticker, nombre_empresa)

    noticias_procesadas = []
    compounds = []
    for noticia in noticias_raw:
        scores = analizar_sentimiento(noticia["titulo"])
        clasificacion = clasificar_sentimiento(scores["compound"])
        compounds.append(scores["compound"])
        noticias_procesadas.append({
            "titulo": noticia["titulo"],
            "publicador": noticia["publicador"],
            "fecha_publicacion": noticia["fecha_publicacion"],
            "fuente": noticia["fuente"],
            "scores": scores,
            "clasificacion": clasificacion,
        })

    score_promedio = round(sum(compounds) / len(compounds), 4) if compounds else 0.0
    clasificacion_general = clasificar_sentimiento(score_promedio)

    documento = {
        "ticker": ticker,
        "empresa": nombre_empresa,
        "cantidad_noticias": len(noticias_procesadas),
        "noticias": noticias_procesadas,
        "score_promedio": limpiar_valor(score_promedio),
        "clasificacion_general": clasificacion_general,
        "created_at": datetime.now()
    }

    # Se borra el registro previo del ticker para evitar duplicados al re-ejecutar
    db["sentimiento_nlp"].delete_many({"ticker": ticker})
    db["sentimiento_nlp"].insert_one(documento)

    print(f"  [{ticker}] {len(noticias_procesadas)} noticias analizadas | "
          f"sentimiento={clasificacion_general} (score={score_promedio:+.4f})")

    return documento

print("Funcion procesar_ticker_nlp definida")


Funcion procesar_ticker_nlp definida


In [10]:
# Paso 10 — Procesar los 5 tickers del proyecto
print("Analizando sentimiento de noticias para los 5 tickers...")
print()

resultados_nlp = {}
for ticker, nombre_empresa in TICKERS.items():
    try:
        resultados_nlp[ticker] = procesar_ticker_nlp(ticker, nombre_empresa)
    except Exception as e:
        print(f"  [{ticker}] Error: {e}")

print()
print("Analisis de sentimiento completo!")


Analizando sentimiento de noticias para los 5 tickers...

  [FSM] 10 noticias analizadas | sentimiento=Bullish (score=+0.3958)
  [VOLCABC1.LM] Noticias reales insuficientes, se completo con respaldo simulado
  [VOLCABC1.LM] 6 noticias analizadas | sentimiento=Bullish (score=+0.2334)
  [ABX.TO] 10 noticias analizadas | sentimiento=Bullish (score=+0.0666)
  [BVN] 10 noticias analizadas | sentimiento=Bullish (score=+0.1256)
  [BHP] 10 noticias analizadas | sentimiento=Bullish (score=+0.0649)

Analisis de sentimiento completo!


In [11]:
# Paso 11 — Celda de verificacion: resultados guardados en MongoDB
print("Resumen de la coleccion sentimiento_nlp:")
print()

resumen_ok = True
for ticker in TICKERS.keys():
    doc = db["sentimiento_nlp"].find_one({"ticker": ticker})
    if doc:
        n = doc["cantidad_noticias"]
        clasif = doc["clasificacion_general"]
        score = doc["score_promedio"]
        print(f"  [OK] {ticker}: {n} noticias | sentimiento={clasif} (score={score:+.4f})")
    else:
        print(f"  [ATENCION] {ticker}: no se encontraron datos")
        resumen_ok = False

print()
if resumen_ok:
    print("Verificacion exitosa: los 5 tickers tienen sentimiento en MongoDB.")
else:
    print("Atencion: revisar los tickers marcados como [ATENCION] antes de continuar.")


Resumen de la coleccion sentimiento_nlp:

  [OK] FSM: 10 noticias | sentimiento=Bullish (score=+0.3958)
  [OK] VOLCABC1.LM: 6 noticias | sentimiento=Bullish (score=+0.2334)
  [OK] ABX.TO: 10 noticias | sentimiento=Bullish (score=+0.0666)
  [OK] BVN: 10 noticias | sentimiento=Bullish (score=+0.1256)
  [OK] BHP: 10 noticias | sentimiento=Bullish (score=+0.0649)

Verificacion exitosa: los 5 tickers tienen sentimiento en MongoDB.


In [12]:
# Paso 12 — Visualizacion: Gauge Chart del sentimiento consolidado
import plotly.graph_objects as go

def grafico_gauge_sentimiento(ticker):
    """Genera un gauge chart con el score de sentimiento promedio de un ticker."""
    doc = db["sentimiento_nlp"].find_one({"ticker": ticker})
    if not doc:
        print(f"  [{ticker}] Sin datos de sentimiento para graficar.")
        return

    score = doc["score_promedio"]
    clasificacion = doc["clasificacion_general"]

    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score,
        number={"valueformat": ".3f"},
        title={"text": f"Sentimiento de noticias — {ticker} ({clasificacion})"},
        gauge={
            "axis": {"range": [-1, 1]},
            "bar": {"color": "black"},
            "steps": [
                {"range": [-1, -0.05], "color": "#e74c3c"},
                {"range": [-0.05, 0.05], "color": "#f1c40f"},
                {"range": [0.05, 1], "color": "#2ecc71"},
            ],
            "threshold": {
                "line": {"color": "black", "width": 3},
                "thickness": 0.8,
                "value": score,
            },
        },
    ))
    fig.update_layout(height=350, margin=dict(t=60, b=10, l=30, r=30))
    fig.show()

# Ejemplo con el primer ticker del proyecto. Puedes llamar la funcion con
# cualquier otro ticker, ej: grafico_gauge_sentimiento("BHP")
primer_ticker = list(TICKERS.keys())[0]
grafico_gauge_sentimiento(primer_ticker)


In [13]:
# Paso 13 — Exportar resultados a JSON local (datos_nlp.json)
import json

def documento_a_serializable(doc):
    """Convierte un documento de MongoDB (con ObjectId/datetime) a un dict serializable en JSON."""
    doc = dict(doc)
    doc.pop("_id", None)
    if isinstance(doc.get("created_at"), datetime):
        doc["created_at"] = doc["created_at"].isoformat()
    return doc

datos_nlp = {}
for ticker in TICKERS.keys():
    doc = db["sentimiento_nlp"].find_one({"ticker": ticker})
    if doc:
        datos_nlp[ticker] = documento_a_serializable(doc)

with open("datos_nlp.json", "w", encoding="utf-8") as f:
    json.dump(datos_nlp, f, indent=2, ensure_ascii=False)

print("Archivo datos_nlp.json generado")


Archivo datos_nlp.json generado


In [14]:
# Paso 14 — Celda de verificacion final: archivo JSON
import os

if os.path.exists("datos_nlp.json") and os.path.getsize("datos_nlp.json") > 0:
    with open("datos_nlp.json", "r", encoding="utf-8") as f:
        contenido = json.load(f)
    print(f"[OK] datos_nlp.json creado correctamente con {len(contenido)} tickers.")
    for ticker, info in contenido.items():
        print(f"  {ticker}: {info['cantidad_noticias']} noticias | {info['clasificacion_general']}")
else:
    print("[ATENCION] El archivo datos_nlp.json no se genero correctamente.")


[OK] datos_nlp.json creado correctamente con 5 tickers.
  FSM: 10 noticias | Bullish
  VOLCABC1.LM: 6 noticias | Bullish
  ABX.TO: 10 noticias | Bullish
  BVN: 10 noticias | Bullish
  BHP: 10 noticias | Bullish


## Resultado

La colección `sentimiento_nlp` contiene el análisis de sentimiento de noticias financieras (metadatos de cada noticia, scores de VADER y clasificación Bullish/Bearish/Neutral) para los 5 tickers del proyecto, y el archivo `datos_nlp.json` queda listo para consumo del frontend.

**Nota:** Este notebook cubre únicamente el módulo de **Análisis de Sentimiento NLP (VADER)** de la Fase 3. El módulo de **LSTM Regressor** (pronóstico de precios) se documentará en un notebook independiente que compartirá la misma base de datos `ernesto_investing_ai`.

**Pipeline:** yfinance (noticias) → limpieza y tokenización → VADER → **MongoDB (`sentimiento_nlp`)** → **JSON (`datos_nlp.json`)** ✓